In [4]:
import pandas as pd

df = pd.read_json("../data/support_tickets.json")
df.head()

,ticket_id,created_at,updated_at,customer_id,customer_tier,organization_id,product,product_version,product_module,category,...,response_count,attachments_count,contains_error_code,contains_stack_trace,business_impact,affected_users,weekend_ticket,after_hours,language,region
0,TK-2024-000001,2023-11-02 12:30:10+00:00,2023-11-02 15:30:46+00:00,CUST-02387,starter,ORG-234,CloudBackup Enterprise,4.5.10,encryption_layer,Feature Request,...,9,4,False,False,high,222,False,False,de,APAC
1,TK-2024-000002,2023-02-10 16:31:31+00:00,2023-02-12 09:59:43+00:00,CUST-03724,free,ORG-435,DataSync Pro,4.1.11,data_validator,Account Management,...,4,0,True,False,medium,18,False,False,ja,MEA
2,TK-2024-000003,2024-09-30 07:43:47+00:00,2024-09-30 11:58:47+00:00,CUST-00600,enterprise,ORG-208,API Gateway,3.1.4,request_router,Feature Request,...,4,5,True,True,medium,591,False,True,ja,NA
3,TK-2024-000004,2024-11-27 18:17:26+00:00,2024-11-30 22:07:50+00:00,CUST-04795,starter,ORG-231,CloudBackup Enterprise,3.4.15,backup_service,Account Management,...,10,4,True,True,critical,34,False,True,en,LATAM
4,TK-2024-000005,2024-03-09 15:41:02+00:00,2024-03-10 10:53:38+00:00,CUST-01101,starter,ORG-241,StreamProcessor,2.8.8,monitoring,Feature Request,...,6,3,True,False,medium,325,True,False,de,MEA


In [5]:
df.describe()

,previous_tickets,resolution_time_hours,resolution_attempts,agent_experience_months,transferred_count,satisfaction_score,account_age_days,account_monthly_value,similar_issues_last_30_days,product_version_age_days,ticket_text_length,response_count,attachments_count,affected_users
count,110000.000000,110000.000000,110000.000000,110000.000000,110000.000000,110000.000000,110000.000000,110000.000000,110000.000000,110000.000000,110000.000000,110000.000000,110000.000000,110000.000000
mean,4.985973,23.913594,3.000782,31.422309,1.500809,3.186282,515.503091,14991.174827,132.029800,90.173918,209.617727,5.510773,2.502055,261.958800
std,3.163863,29.930534,1.415042,15.979575,1.117071,1.294954,281.085669,25515.529007,155.585577,51.985596,43.136495,2.882536,1.707333,313.407295
min,0.000000,0.210000,1.000000,3.000000,0.000000,1.000000,30.000000,0.000000,0.000000,1.000000,157.000000,1.000000,0.000000,1.000000
25%,2.000000,3.820000,2.000000,18.000000,1.000000,2.000000,267.000000,85.000000,52.000000,45.000000,167.000000,3.000000,1.000000,24.000000
50%,5.000000,11.720000,3.000000,33.000000,1.000000,3.000000,515.000000,1335.000000,105.000000,90.000000,219.000000,6.000000,3.000000,48.000000
75%,8.000000,32.830000,4.000000,43.000000,3.000000,4.000000,760.000000,16396.000000,158.000000,135.000000,235.000000,8.000000,4.000000,497.000000
max,10.000000,214.270000,5.000000,60.000000,3.000000,5.000000,1000.000000,99964.000000,1000.000000,180.000000,316.000000,10.000000,5.000000,1000.000000


Let's explore the target variable category and subcategory

In [6]:
df.category.unique()

<StringArray>
[   'Feature Request', 'Account Management',           'Security',
         'Data Issue',    'Technical Issue']
Length: 5, dtype: str

In [7]:
df.subcategory.unique()

<StringArray>
[ 'Documentation',        'Upgrade',    'New Feature',            'API',
     'Compliance',     'Corruption',    'Enhancement',     'Sync Error',
          'UI/UX',  'Import/Export',            'Bug', 'Access Control',
  'Vulnerability', 'Authentication',        'Billing',    'Integration',
  'Configuration',    'Performance',     'Encryption',        'License',
  'Authorization',  'Compatibility',   'Subscription',     'Validation',
      'Data Loss']
Length: 25, dtype: str

In [8]:
df[["category", "subcategory"]].value_counts(normalize=True).sort_index() * 100

category            subcategory   
Account Management  Access Control    4.016364
                    Billing           4.061818
                    License           4.006364
                    Subscription      3.990000
                    Upgrade           3.922727
Data Issue          Corruption        4.083636
                    Data Loss         3.954545
                    Import/Export     4.028182
                    Sync Error        4.005455
                    Validation        3.973636
Feature Request     API               3.994545
                    Documentation     3.991818
                    Enhancement       3.999091
                    New Feature       3.971818
                    UI/UX             4.085455
Security            Authentication    4.072727
                    Authorization     4.050000
                    Compliance        4.060909
                    Encryption        3.975455
                    Vulnerability     3.918182
Technical Issue     Bug  

There are some variables that quite likely will not help with the classification. We remove those : ticket_id, created_at, updated_at, previous_tickets, customer_id, channel

In [10]:
df = df.drop(['ticket_id', 'created_at', 'updated_at', 'customer_id', 'channel'], axis=1)

There are some variables that will likely be very important to find the category : product, product_version, product_module, priority, severity, subject, description, error_logs, stack_trace

There are some that we are not sure :

In [ ]:
import pandas as pd

# Count how many distinct categories each subcategory belongs to
result = df.groupby('subcategory')['category'].nunique().reset_index()
result.columns = ['subcategory', 'category_count']
result